<a href="https://colab.research.google.com/github/simon-clematide/colab-notebooks-for-teaching/blob/main/spam_or_ham_sms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📩 Welcome to the SMS Spam Classifier Tutorial!

Have you ever wondered how your phone automatically moves annoying promotional texts into a "Junk" or "Spam" folder? Today, you are going to build exactly that!

In this beginner-friendly tutorial, we will use the famous **SMS Spam Collection Dataset** to teach a machine learning model how to distinguish between normal text messages ("Ham") and unwanted commercial messages ("Spam").

### What we will do:
1. **Load and Explore:** Look at real SMS text messages.
2. **Feature Extraction:** Convert English words into mathematical patterns a computer can understand (using a basic Natural Language Processing technique called Bag-of-Words).
3. **Train the Brain:** Teach a **Naive Bayes** machine learning model to spot the difference between Spam and Ham.
4. **Test the Model:** Give the model brand-new messages to see how accurate it really is!

No prior coding or math experience is needed to understand the concepts. Let's get started! 🚀

## Step 1: Gather the Tools and Raw Data
Before we can build our spam filter, we need to gather our tools and the raw material (the dataset). We'll download the SMS Spam Collection, which contains thousands of real text messages.

In [ ]:
# @title Step 1: Prepare the tools and data { display-mode: "form" }

# Import the libraries we need
import pandas as pd
import urllib.request
import zipfile
import os

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score


# Download the SMS dataset (only if it is not already available)
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00228/smsspamcollection.zip"

if not os.path.exists("SMSSpamCollection"):
    print("Downloading the SMS dataset...")

    urllib.request.urlretrieve(url, "sms.zip")

    with zipfile.ZipFile("sms.zip") as file:
        file.extract("SMSSpamCollection")

print("Setup complete!")

## Step 2: Look at the Data
Now that we have the file, we need to open it and look inside. We load the messages into a virtual table so our program can easily read and manipulate them.

In [ ]:
# Load the SMS messages
df = pd.read_csv(
    "SMSSpamCollection",
    sep="\t",
    names=["label", "message"]
)

# Display 10 randomly selected messages
display(df.sample(10))

### Step 2.5: Subsample to Balance the Dataset
Our dataset has a strong class imbalance: there are 4,827 legitimate messages ("ham") and only 747 spam messages. If we kept it this way, a lazy model that predicts "ham" 100% of the time would still get ~86.6% accuracy!

To prevent this, we'll subsample our dataset so we have an equal number of ham and spam messages (747 of each).

In [ ]:
# Step 2.5: Balance the dataset

# Separate spam and ham messages
spam_df = df[df["label"] == "spam"]
ham_df = df[df["label"] == "ham"]

# Randomly select the same number of ham messages as spam messages
ham_sample = ham_df.sample(n=len(spam_df))

# Combine and shuffle the messages
df = pd.concat([spam_df, ham_sample])
df = df.sample(frac=1).reset_index(drop=True)


# Count spam and ham messages
counts = df["label"].value_counts()

# Display counts and percentages
stats = pd.DataFrame({
    "Messages": counts,
    "Percentage": (counts / counts.sum() * 100).round(1)
})

display(stats)

## Step 3: Prepare the Data (Train, Dev, and Test)

First, we convert our category labels into numbers:

- **Ham = 0**
- **Spam = 1**

Then, we randomly divide our balanced dataset into three groups:

- **Training set (80%) — The Textbook:** The model learns
  statistical patterns from these messages.

- **Development set (10%) — The Practice Quiz:** We compare
  different settings, such as whether to convert words to lowercase.

- **Test set (10%) — The Final Exam:** We evaluate the final
  model on messages that were not used for training or
  selecting its settings.

#### Stratified random splitting

We use **stratification** to preserve approximately the same
proportion of spam and ham in all three datasets.

The messages are still selected randomly. If we repeat the
experiment, we may obtain different results.

**Important:** Because we balanced the original dataset,
our test results describe performance on a balanced collection
of messages, not necessarily on SMS messages encountered
in everyday life.

In [ ]:
# Step 3: Prepare the training, development, and test data

# Convert labels into numbers: ham = 0, spam = 1
df["label_num"] = df["label"].map({"ham": 0, "spam": 1})

# Separate messages (X) and their labels (y)
X = df["message"]
y = df["label_num"]

# Split into training (80%) and remaining data (20%)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.2, stratify=y
)

# Split the remaining data into development (10%) and test (10%)
X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp
)

# Show the size of each dataset
print("Training messages:", len(X_train))
print("Development messages:", len(X_dev))
print("Test messages:", len(X_test))

### Let Compare the three datasets

In [ ]:
# Count ham (0) and spam (1) messages in each dataset
counts = pd.DataFrame({
    "Training": y_train.value_counts(),
    "Development": y_dev.value_counts(),
    "Test": y_test.value_counts()
})

# Replace numerical labels with readable names
counts.index = counts.index.map({0: "Ham", 1: "Spam"})

# Display the counts
display(counts)

# Display the proportions
print("\nPercentage of messages in each category:")
display((counts / counts.sum() * 100).round(1))

## 🧠 How Naive Bayes Works: Two Urns of Words

Our classifier uses **Multinomial Naive Bayes**.

The central idea is to imagine that each category generates
messages by drawing words from its own urn.

### 1. Training: Fill the urns

Imagine two urns:

- **HAM:** Contains words from legitimate messages.
- **SPAM:** Contains words from spam messages.

During training, we count how often each word occurs in each category.

The probability of drawing word $w$ from category $c$ is:

$$
P(w\mid c)=\frac{\operatorname{count}(w,c)}{N_c}
$$

Here, $N_c$ is the total number of word occurrences in category $c$.

### 2. Generating a message: Draw words from an urn

Imagine generating a message by repeatedly drawing words
from one of the urns.

We assume that:

- Words are drawn independently, with replacement.
- Word order and context are ignored (**Bag-of-Words**).

These assumptions simplify how we model text.

### 3. Inference: Which urn could have generated the message?

For a new message containing words $w_1,\ldots,w_n$,
we calculate how likely each urn is to generate those words:

$$
S_c=\prod_{i=1}^{n}P(w_i\mid c)
$$

We calculate one likelihood score for HAM and one for SPAM.

### 4. Decision: Which category is more likely?

With equal category priors and equal error costs,
we simply choose the urn with the higher likelihood:

$$
\text{Prediction}=
\begin{cases}
\text{spam} & \text{if } S_{\text{spam}}>S_{\text{ham}}\\
\text{ham} & \text{otherwise}
\end{cases}
$$

Our implementation also takes into account how common each
category is in the training data.

Because we balanced the dataset, these starting probabilities
are approximately equal.

---

**Key idea:** Naive Bayes learns which words are characteristic
of each category and uses their probabilities to classify new messages.

It does not understand the meaning of the messages.

### Problem: One zero ruins everything!

If a word has never appeared in one category,
its estimated probability is zero.

Multiplying by zero makes the entire likelihood score zero.

### Solution: Laplace smoothing

We start every word in the shared vocabulary
with a count of 1 instead of 0.

$$
P(w\mid c)=
\frac{\operatorname{count}(w,c)+1}{N_c+|V|}
$$

Here, $|V|$ is the number of distinct words in our vocabulary.

This prevents zero probabilities for words in the vocabulary.

Words that are completely unknown to the vectorizer are ignored.



In [ ]:
# Demonstration: How does Bag-of-Words represent a message?

message = ["free prize free money"]

# Convert the message into word counts
vectorizer = CountVectorizer()
word_counts = vectorizer.fit_transform(message)

# Display the result
bow = pd.DataFrame(
    word_counts.toarray(),
    columns=vectorizer.get_feature_names_out()
)

display(bow)

## Step 4: The Experiment — Does Lowercasing Help?

Should we convert all text to lowercase before training our classifier?

Consider these examples:

- `FREE` and `free`
- `WIN` and `win`

If we use **lowercasing**, the model treats each pair as the same token.

If we **preserve capitalization**, the model treats them as different tokens.

Both approaches have potential advantages:

- **Lowercasing:** The model can combine evidence from different capitalizations of the same word.
- **Preserving capitalization:** The model can use capitalization as a possible indicator of spam.

### Let's test it!

We train two classifiers that differ only in their lowercase setting:

1. `lowercase=True`
2. `lowercase=False`

We compare their accuracy on the **development set**.

**Before running the experiment:**

Which setting do you expect to perform better? Why?

**Remember:** A difference in accuracy on one development set does not establish that one approach is always better.

In [ ]:
# Step 4: Does lowercasing improve classification?

def test_lowercasing(lowercase_setting):

    # Convert messages into numerical features
    vectorizer = CountVectorizer(
        lowercase=lowercase_setting,
        max_features=2000,
        token_pattern=r"(?u)\w+|[^\w\s]+"
    )

    X_train_vec = vectorizer.fit_transform(X_train)
    X_dev_vec = vectorizer.transform(X_dev)

    # Train the classifier
    model = MultinomialNB()
    model.fit(X_train_vec, y_train)

    # Evaluate on the development set
    predictions = model.predict(X_dev_vec)
    return accuracy_score(y_dev, predictions)


# Run the experiment with two settings
acc_true = test_lowercasing(True)
acc_false = test_lowercasing(False)

# Compare the results
results = pd.DataFrame({
    "Lowercase": [True, False],
    "Development accuracy": [acc_true, acc_false]
})

display(results.style.format({"Development accuracy": "{:.1%}"}))

## Step 5: Train the Final Model

We have compared two ways of processing text and selected the one that performed better on our **development set**.

Now we train our final classifier using this setting.

The model learns from the **training data**. We keep the **test data** separate so that we can evaluate the model on messages it has never seen.

**Remember:** The development set helps us make decisions about the model. The test set tells us how well our final model performs on unseen data.

In [ ]:
# Step 5: Train the final model

# Convert the training messages into numerical features
final_vectorizer = CountVectorizer(
    lowercase=LOWERCASE_SETTING,
    max_features=2000,
    token_pattern=r"(?u)\w+|[^\w\s]+"
)

X_train_vec_final = final_vectorizer.fit_transform(X_train)

# Train the Naive Bayes classifier
final_model = MultinomialNB()
final_model.fit(X_train_vec_final, y_train)

print(f"Final model trained with lowercase={LOWERCASE_SETTING}")

## Step 6: The Final Exam (Test Set)

We have trained our classifier and selected its settings using
the development set.

Now we evaluate it on the **test set**: messages that were
not used for training or selecting the model's settings.

We measure **accuracy**: the percentage of messages
classified correctly.

This gives us an estimate of how well the classifier
performs on unseen messages.

**Remember:**
- A different random sample may produce a different accuracy.
- Our test set is balanced, so its results do not necessarily
  reflect performance on everyday SMS messages.

In [ ]:
# Step 6: Evaluate the model on unseen test messages

# Convert test messages into numerical features
X_test_vec = final_vectorizer.transform(X_test)

# Predict whether each message is spam or ham
y_test_pred = final_model.predict(X_test_vec)

# Compare predictions with the correct labels
accuracy = accuracy_score(y_test, y_test_pred)

print(f"Test accuracy: {accuracy:.1%}")

## Step 7: Try It Yourself!

Our classifier is ready! Let's see how it classifies some new SMS messages.

**Before running the code:**

- Which messages do you expect to be classified as **spam**?
- Which messages do you expect to be classified as **ham**?
- Are there any messages that might confuse the classifier?

Now run the code and compare your predictions with the model's results.

### Experiment!

Try changing one of the messages and running the code again.

**Can you find a legitimate message that the classifier predicts as spam, or a spam message that it predicts as ham?**

Remember: The classifier uses statistical patterns learned from the training data. Its predictions are not always correct.

In [ ]:
# Step 7: Classify new messages

# Try changing these messages!
messages = [
    "Hey, are we still on for dinner tonight?",
    "Congratulations! You have won a free prize!",
    "Don't forget to bring the documents to the meeting.",
    "URGENT: Your account has been suspended. Reply with your PIN.",
    "Free pizza in the department kitchen!"
]

# Convert messages into numerical features
message_vectors = final_vectorizer.transform(messages)

# Predict spam (1) or ham (0)
predictions = final_model.predict(message_vectors)

# Display the results
results = pd.DataFrame({
    "Message": messages,
    "Prediction": predictions
})

results["Prediction"] = results["Prediction"].map({
    0: "HAM",
    1: "SPAM"
})

display(results)

## Step 8: Peek Inside the Model — Which Words Matter?

Remember our **two urns of words**?

During training, Naive Bayes learned how probable each token is
in spam and ham messages.

We can now compare these probabilities to find:

- **Tokens associated with spam:** More probable in the spam urn.
- **Tokens associated with ham:** More probable in the ham urn.

We calculate a score for each token:

$$
\text{score}(w) =
\log P(w\mid\text{spam}) -
\log P(w\mid\text{ham})
$$

- **Positive score:** Stronger association with spam.
- **Negative score:** Stronger association with ham.
- **Score near zero:** Similar probabilities in both categories.

The larger the absolute score, the stronger the association.

### Let's inspect what the classifier learned!

Which tokens do you expect to be most strongly associated
with spam? Which with ham?

**Remember:** These are statistical associations learned from
the training messages, not the meanings of the words.

In [ ]:
# Step 8: Which tokens are associated with spam and ham?

# Get the tokens in the model's vocabulary
tokens = final_vectorizer.get_feature_names_out()

# Compare the probability of each token in spam and ham
spam_scores = (
    final_model.feature_log_prob_[1]
    - final_model.feature_log_prob_[0]
)

# Create a table of tokens and their scores
feature_df = pd.DataFrame({
    "Token": tokens,
    "Spam score": spam_scores
})

In [ ]:
# Tokens most strongly associated with spam
print("Top 15 tokens associated with SPAM:")

display(
    feature_df.sort_values("Spam score", ascending=False).head(15)
)

# Tokens most strongly associated with ham
print("Top 15 tokens associated with HAM:")

display(
    feature_df.sort_values("Spam score").head(15)
)